In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA06 - Donations
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit, either directly or through the use of 
   third parties, make any domestic/international charitable donations?
   
   LOGIC SUMMARY:
   1. MASTER AU ANCHOR: Uses the Master AU list as the absolute key for the output. 
      Only AUs in this filtered list (Canada, HK, Barbados + US_OR_CANADA = 'CANADA') 
      will appear in the final report.
   2. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is entirely excluded 
      based on the business logic.
   3. CATEGORY CODE FILTER: Strictly filters the Finance data for Category Codes 
      '292' and '427' (Charitable Donations).
   4. MAPPING & OUTPUT: Maps flagged transactions to AU_IDs using the CC Bootstrap 
      (with safe Integer casting on cost_center_id). LEFT JOINS this mapped data back 
      to the Master AU anchor and calculates findings divided by total mapped records.
=================================================================================== */

WITH Master_AUs AS (
    -- 1. Grab Master List data: This is our strict anchor for the final output
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    -- 2. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    -- 3. Clean the Finance columns for the percentage denominator population
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Donation_Transactions AS (
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE CatCode IN ('292', '427')
),

CC_Mapping AS (
    -- Standardize the CC Mapping table using the correct cost_center_id column
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Employee_Roster AS (
    SELECT 
        CASE WHEN TRIM(DeptID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(DeptID), 4), 4, '0') ELSE UPPER(TRIM(DeptID)) END AS DeptID,
        TRIM(FullName) AS FullName
    FROM hive_metastore.ra_adido_2025.fy25_mltf_sanctions_and_abac_11282025_all_employees
    WHERE DeptID IS NOT NULL
      AND FullName IS NOT NULL
      AND TRIM(FullName) != ''
),

Employee_Counts AS (
    SELECT 
        m.AU_ID,
        COUNT(DISTINCT e.FullName) AS Total_Employees
    FROM Employee_Roster e
    INNER JOIN CC_Mapping m
        ON e.DeptID = m.Mapped_CC
    GROUP BY m.AU_ID
),

Flagged_AUs AS (
    -- Map the donation transactions to their respective AU_IDs
    SELECT 
        m.AU_ID,
        COUNT(*) AS Flagged_Count
    FROM Donation_Transactions d
    INNER JOIN CC_Mapping m
        ON d.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
),

Total_Records_By_AU AS (
    SELECT 
        m.AU_ID,
        COUNT(*) AS Total_Record_Count
    FROM All_Finance_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
    GROUP BY m.AU_ID
)

-- 4. Final Output: Strictly structured to the requested 6 columns anchored to Master list
SELECT 
    a.BusinessID,                          
    a.AU_Name,                             
    a.Business_Segment,
    'EBA06' AS QuestionID,               
    CASE 
        WHEN COALESCE(r.Total_Record_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(f.Flagged_Count, 0) AS DOUBLE) / r.Total_Record_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    a.In_ABAC_AU_List
    
FROM Master_AUs a
LEFT JOIN Flagged_AUs f
    ON a.BusinessID = f.AU_ID
LEFT JOIN Total_Records_By_AU r
    ON a.BusinessID = r.AU_ID
ORDER BY a.BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA06 - AU Level Calculation Review
   PURPOSE: One row per AU showing normalized Cost Centers and how the final percentage
   response was calculated.
=================================================================================== */

WITH Master_AUs AS (
    SELECT 
        TRIM(CAST(BusinessID AS STRING)) AS BusinessID,
        TRIM(Business_Operational_Unit_Name) AS AU_Name,
        TRIM(BusinessSegment) AS Business_Segment,
        'Yes' AS In_ABAC_AU_List
    FROM hive_metastore.ra_adido_2025.fy25_list_of_units
    WHERE BusinessID IS NOT NULL
      AND UPPER(TRIM(Jurisdiction)) IN ('CANADA', 'HONG KONG', 'BARBADOS')
      AND UPPER(TRIM(US_OR_CANADA)) = 'CANADA'
),

Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

All_Finance_Transactions AS (
    SELECT 
        CASE WHEN TRIM(CostCenter) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(CostCenter), 4), 4, '0') ELSE UPPER(TRIM(CostCenter)) END AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE CostCenter IS NOT NULL
),

Donation_Transactions AS (
    SELECT Cleaned_CC, CatCode
    FROM All_Finance_Transactions
    WHERE CatCode IN ('292', '427')
),

CC_Mapping AS (
    SELECT DISTINCT 
        TRIM(CAST(AU_ID AS STRING)) AS AU_ID,
        CASE WHEN TRIM(cost_center_id) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(cost_center_id), 4), 4, '0') ELSE UPPER(TRIM(cost_center_id)) END AS Mapped_CC
    FROM vw_cost_center_mapping_bootstrap
    WHERE AU_ID IS NOT NULL AND cost_center_id IS NOT NULL
),

Employee_Roster AS (
    SELECT 
        CASE WHEN TRIM(DeptID) RLIKE '^[0-9]+$' THEN LPAD(RIGHT(TRIM(DeptID), 4), 4, '0') ELSE UPPER(TRIM(DeptID)) END AS DeptID,
        TRIM(FullName) AS FullName
    FROM hive_metastore.ra_adido_2025.fy25_mltf_sanctions_and_abac_11282025_all_employees
    WHERE DeptID IS NOT NULL
      AND FullName IS NOT NULL
      AND TRIM(FullName) != ''
),

Employee_Stats_By_AU AS (
    SELECT 
        m.AU_ID,
        COUNT(DISTINCT e.FullName) AS Total_Employees,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(e.DeptID))) AS Employee_DeptID_List
    FROM Employee_Roster e
    INNER JOIN CC_Mapping m
        ON e.DeptID = m.Mapped_CC
    GROUP BY m.AU_ID
),

Mapped_All_Finance AS (
    SELECT 
        m.AU_ID,
        f.Cleaned_CC,
        f.CatCode
    FROM All_Finance_Transactions f
    INNER JOIN CC_Mapping m
        ON f.Cleaned_CC = m.Mapped_CC
),

Mapped_Donations AS (
    SELECT 
        m.AU_ID,
        d.Cleaned_CC,
        d.CatCode
    FROM Donation_Transactions d
    INNER JOIN CC_Mapping m
        ON d.Cleaned_CC = m.Mapped_CC
),

Donation_By_AU AS (
    SELECT 
        AU_ID,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(Cleaned_CC))) AS Normalized_CC_List,
        CONCAT_WS(', ', SORT_ARRAY(COLLECT_SET(CatCode))) AS Donation_CatCode_List,
        COUNT(*) AS Donation_Row_Count
    FROM Mapped_Donations
    GROUP BY AU_ID
),

Finance_Record_By_AU AS (
    SELECT 
        AU_ID,
        COUNT(*) AS Mapped_Row_Count
    FROM Mapped_All_Finance
    GROUP BY AU_ID
)

SELECT 
    mast.BusinessID,
    mast.AU_Name,
    mast.Business_Segment,
    'EBA06' AS QuestionID,
    CASE 
        WHEN COALESCE(rec.Mapped_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Donation_Row_Count, 0) AS DOUBLE) / rec.Mapped_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Response,
    COALESCE(d.Normalized_CC_List, 'n/a') AS Normalized_CC_List,
    COALESCE(d.Donation_CatCode_List, 'n/a') AS Donation_CatCode_List,
    COALESCE(rec.Mapped_Row_Count, 0) AS Mapped_Row_Count,
    COALESCE(d.Donation_Row_Count, 0) AS Donation_Row_Count,
    COALESCE(d.Donation_Row_Count, 0) AS Numerator,
    COALESCE(rec.Mapped_Row_Count, 0) AS Denominator,
    CASE 
        WHEN COALESCE(rec.Mapped_Row_Count, 0) = 0 THEN '0%'
        ELSE CAST(ROUND((CAST(COALESCE(d.Donation_Row_Count, 0) AS DOUBLE) / rec.Mapped_Row_Count) * 100, 2) AS STRING) || '%'
    END AS Calculated_Percentage,
    CASE 
        WHEN COALESCE(rec.Mapped_Row_Count, 0) = 0 THEN CONCAT('Response = 0% because denominator = 0. Findings = ', CAST(COALESCE(d.Donation_Row_Count, 0) AS STRING), ' and this AU has no mapped source records.')
        ELSE CONCAT('Response = ', CAST(ROUND((CAST(COALESCE(d.Donation_Row_Count, 0) AS DOUBLE) / rec.Mapped_Row_Count) * 100, 2) AS STRING), '% because findings ', CAST(COALESCE(d.Donation_Row_Count, 0) AS STRING), ' / total mapped records ', CAST(rec.Mapped_Row_Count AS STRING), ' * 100.')
    END AS Calculation_Formula,
    mast.In_ABAC_AU_List
FROM Master_AUs mast
LEFT JOIN Donation_By_AU d
    ON mast.BusinessID = d.AU_ID
LEFT JOIN Finance_Record_By_AU rec
    ON mast.BusinessID = rec.AU_ID
ORDER BY mast.BusinessID;
